# 프로젝트 3 - Weekend 1: 도구 호출 기반 법률 검색 기초

**프로젝트**: 법률 문서 기반 검색 에이전트 시스템

**학습 목표**:
1. 법률 도메인 데이터를 `Document`로 구조화하고 FAISS 벡터 스토어로 색인
2. LangChain `@tool` 데코레이터로 법률 검색·해석·비교 도구 구현
3. `bind_tools()` + `ToolMessage`로 LLM이 도구를 자동 선택하는 에이전트 루프 구축
4. 대화형 `LegalChatbot` 클래스와 에러 처리/면책 조항으로 실전급 시스템 완성

**구성**: 문제 10개 × 약 45분 = 약 8시간

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    pandas numpy matplotlib python-dotenv tiktoken

In [ ]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

# ⚠️ LangChain 패턴만 사용 — 원시 openai SDK 직접 import 금지
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

---
## 실습 데이터: 법률 조문 샘플

아래 셀을 실행하면 민법/형법/상법 조문 18개가 로드됩니다. 문제 풀이 내내 이 데이터를 사용합니다.

In [ ]:
# 실습용 법률 조문 샘플 (민법/형법/상법 발췌)
SAMPLE_LAW_ARTICLES = [
    # --- 민법 ---
    {"law": "민법", "article": "제2조", "title": "신의성실의 원칙",
     "content": "권리의 행사와 의무의 이행은 신의에 좇아 성실히 하여야 한다. 권리는 남용하지 못한다.",
     "topic": "기본원칙"},
    {"law": "민법", "article": "제103조", "title": "반사회질서의 법률행위",
     "content": "선량한 풍속 기타 사회질서에 위반한 사항을 내용으로 하는 법률행위는 무효로 한다.",
     "topic": "법률행위"},
    {"law": "민법", "article": "제109조", "title": "착오로 인한 의사표시",
     "content": "의사표시는 법률행위의 내용의 중요부분에 착오가 있는 때에는 취소할 수 있다. 그러나 그 착오가 표의자의 중대한 과실로 인한 때에는 취소하지 못한다.",
     "topic": "의사표시"},
    {"law": "민법", "article": "제390조", "title": "채무불이행과 손해배상",
     "content": "채무자가 채무의 내용에 좇은 이행을 하지 아니한 때에는 채권자는 손해배상을 청구할 수 있다. 그러나 채무자의 고의나 과실없이 이행할 수 없게 된 때에는 그러하지 아니하다.",
     "topic": "채권"},
    {"law": "민법", "article": "제750조", "title": "불법행위의 내용",
     "content": "고의 또는 과실로 인한 위법행위로 타인에게 손해를 가한 자는 그 손해를 배상할 책임이 있다.",
     "topic": "불법행위"},
    {"law": "민법", "article": "제840조", "title": "재판상 이혼원인",
     "content": "부부의 일방은 다음 각호의 사유가 있는 경우에는 가정법원에 이혼을 청구할 수 있다. 1. 배우자에 부정한 행위가 있었을 때 2. 배우자가 악의로 다른 일방을 유기한 때 3. 배우자 또는 그 직계존속으로부터 심히 부당한 대우를 받았을 때 4. 자기의 직계존속이 배우자로부터 심히 부당한 대우를 받았을 때 5. 배우자의 생사가 3년이상 분명하지 아니한 때 6. 기타 혼인을 계속하기 어려운 중대한 사유가 있을 때",
     "topic": "가족법"},
    # --- 형법 ---
    {"law": "형법", "article": "제250조", "title": "살인, 존속살해",
     "content": "사람을 살해한 자는 사형, 무기 또는 5년 이상의 징역에 처한다. 자기 또는 배우자의 직계존속을 살해한 자는 사형, 무기 또는 7년 이상의 징역에 처한다.",
     "topic": "생명"},
    {"law": "형법", "article": "제260조", "title": "폭행, 존속폭행",
     "content": "사람의 신체에 대하여 폭행을 가한 자는 2년 이하의 징역, 500만원 이하의 벌금, 구류 또는 과료에 처한다. 자기 또는 배우자의 직계존속에 대하여 폭행을 가한 자는 5년 이하의 징역 또는 700만원 이하의 벌금에 처한다.",
     "topic": "신체"},
    {"law": "형법", "article": "제329조", "title": "절도",
     "content": "타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다.",
     "topic": "재산"},
    {"law": "형법", "article": "제347조", "title": "사기",
     "content": "사람을 기망하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 10년 이하의 징역 또는 2천만원 이하의 벌금에 처한다. 전항의 방법으로 제삼자로 하여금 재물의 교부를 받게 하거나 재산상의 이익을 취득하게 한 때에도 전항의 형과 같다.",
     "topic": "재산"},
    {"law": "형법", "article": "제366조", "title": "재물손괴등",
     "content": "타인의 재물, 문서 또는 전자기록등 특수매체기록을 손괴 또는 은닉 기타 방법으로 기효용을 해한 자는 3년 이하의 징역 또는 700만원 이하의 벌금에 처한다.",
     "topic": "재산"},
    {"law": "형법", "article": "제307조", "title": "명예훼손",
     "content": "공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 금고 또는 500만원 이하의 벌금에 처한다. 공연히 허위의 사실을 적시하여 사람의 명예를 훼손한 자는 5년 이하의 징역, 10년 이하의 자격정지 또는 1천만원 이하의 벌금에 처한다.",
     "topic": "명예"},
    # --- 상법 ---
    {"law": "상법", "article": "제46조", "title": "기본적 상행위",
     "content": "영업으로 하는 다음의 행위를 상행위라 한다. 다만, 오로지 임금을 받을 목적으로 물건을 제조하거나 노무에 종사하는 자의 행위는 그러하지 아니하다. 1. 동산, 부동산, 유가증권 기타의 재산의 매매 2. 동산, 부동산, 유가증권 기타의 재산의 임대차 ... (22호)",
     "topic": "상행위"},
    {"law": "상법", "article": "제169조", "title": "회사의 의의",
     "content": "이 법에서 회사란 상행위나 그 밖의 영리를 목적으로 하여 설립한 법인을 말한다.",
     "topic": "회사법"},
    {"law": "상법", "article": "제329조", "title": "자본금의 구성",
     "content": "주식회사의 자본금은 이 법에서 달리 규정한 경우 외에는 발행주식의 액면총액으로 한다. 회사는 정관으로 정한 경우에는 주식의 전부를 무액면주식으로 발행할 수 있다. 다만, 무액면주식을 발행하는 경우에는 액면주식을 발행할 수 없다.",
     "topic": "회사법"},
    {"law": "상법", "article": "제382조", "title": "이사의 선임, 임기",
     "content": "이사는 주주총회에서 선임한다. 이사의 임기는 3년을 초과하지 못한다.",
     "topic": "회사법"},
    {"law": "상법", "article": "제399조", "title": "이사의 회사에 대한 책임",
     "content": "이사가 고의 또는 과실로 법령 또는 정관에 위반한 행위를 하거나 그 임무를 게을리한 때에는 그 이사는 회사에 대하여 연대하여 손해를 배상할 책임이 있다.",
     "topic": "회사법"},
    {"law": "상법", "article": "제732조", "title": "보험계약",
     "content": "보험계약은 당사자 일방이 약정한 보험료를 지급하고 상대방이 일정한 사유가 생길 경우에 일정한 보험금액 기타의 급여를 지급할 것을 약정함으로써 효력이 생긴다.",
     "topic": "보험"},
]

print(f"📚 법률 조문 로드 완료: 총 {len(SAMPLE_LAW_ARTICLES)}개")
laws = set(a["law"] for a in SAMPLE_LAW_ARTICLES)
for law in sorted(laws):
    cnt = sum(1 for a in SAMPLE_LAW_ARTICLES if a["law"] == law)
    print(f"  - {law}: {cnt}개")

---
## 문제 1: 법률 데이터셋 구축 — Document 객체 변환

`SAMPLE_LAW_ARTICLES` 리스트를 LangChain `Document` 객체 리스트로 변환하세요.

**요구사항:**
- `page_content`에는 `"{법령} {조항} ({제목}): {본문}"` 형식의 문자열
- `metadata`에는 `law`, `article`, `title`, `topic` 필드 포함
- 함수 시그니처: `build_law_documents(articles: list) -> list[Document]`

**평가기준:**
- 반환된 리스트 길이 = 18
- 각 Document가 `page_content`와 `metadata` 모두 가짐
- metadata에 위 4개 필드 존재

In [ ]:
# 문제 1: 법률 조문을 Document 객체로 변환
def build_law_documents(articles):
    """법률 조문 dict 리스트를 Document 리스트로 변환합니다."""
    documents = []
    for article in articles:
        # ---- 여기에 코드 작성 ----
        # 1) page_content 구성: "{법령} {조항} ({제목}): {본문}"
        # 2) metadata 구성: law, article, title, topic
        # 3) Document(page_content=..., metadata=...) 생성 후 append
        pass
    return documents

law_docs = build_law_documents(SAMPLE_LAW_ARTICLES)
print(f"✅ Document {len(law_docs)}개 생성")
print(f"첫 문서 page_content: {law_docs[0].page_content[:80] if law_docs else 'N/A'}...")
print(f"첫 문서 metadata: {law_docs[0].metadata if law_docs else 'N/A'}")

---
## 문제 2: FAISS 벡터스토어 구축

문제 1에서 만든 `law_docs`를 FAISS 벡터스토어로 색인하세요.

**요구사항:**
- `OpenAIEmbeddings(model="text-embedding-3-small")` 사용
- `FAISS.from_documents()` 활용
- `similarity_search(query, k=3)`로 테스트 쿼리 검색 확인
- 함수 시그니처: `build_vector_store(docs: list[Document]) -> FAISS`

**평가기준:**
- 반환값이 FAISS 인스턴스
- `"이혼 사유"` 쿼리 검색 시 `제840조` 상위 노출

In [ ]:
# 문제 2: FAISS 벡터스토어 구축
def build_vector_store(docs):
    """Document 리스트로 FAISS 벡터스토어를 생성합니다."""
    # ---- 여기에 코드 작성 ----
    # FAISS.from_documents() 사용
    vector_store = None
    return vector_store

vector_store = build_vector_store(law_docs)

# 검색 테스트
query = "이혼 사유"
results = vector_store.similarity_search(query, k=3) if vector_store else []
print(f"🔍 '{query}' 검색 결과 ({len(results)}개):")
for i, doc in enumerate(results, 1):
    meta = doc.metadata
    print(f"  [{i}] {meta.get('law')} {meta.get('article')} - {meta.get('title')}")

---
## 문제 3: @tool `search_law` — 법률 검색 도구

LLM이 호출할 수 있는 법률 검색 도구를 `@tool` 데코레이터로 구현하세요.

**요구사항:**
- 함수명: `search_law`
- 파라미터: `query: str, top_k: int = 3, law_filter: str = ""`
- `law_filter`가 비어있지 않으면 해당 법령으로 필터링 (예: `"민법"`만)
- 검색 결과를 JSON 형식 문자열로 반환 (law, article, title, snippet 포함)
- docstring이 LLM 설명이 되므로 명확히 작성

**평가기준:**
- `search_law.name == "search_law"`
- `search_law.description`에 "법률", "검색" 키워드 포함
- `law_filter="민법"` 시 민법 조문만 반환

In [ ]:
# 문제 3: @tool search_law
@tool
def search_law(query: str, top_k: int = 3, law_filter: str = "") -> str:
    """법률 조문을 의미 기반으로 검색합니다.
    질문과 관련된 법률 조문을 찾아 반환합니다.

    Args:
        query: 검색 질의 (예: "이혼 사유", "명예훼손 처벌")
        top_k: 반환할 조문 수 (기본 3)
        law_filter: 특정 법령으로 필터링 ("민법", "형법", "상법" 중 하나, 빈 문자열이면 전체)
    """
    # ---- 여기에 코드 작성 ----
    # 1) vector_store.similarity_search(query, k=top_k*3) 로 여유있게 검색
    # 2) law_filter가 비어있지 않으면 metadata['law']로 필터링
    # 3) 상위 top_k개만 선택
    # 4) JSON 문자열로 변환하여 반환
    return "[]"

# 테스트
print("도구 이름:", search_law.name)
print("도구 설명:", search_law.description[:80], "...")
result = search_law.invoke({"query": "재산 범죄", "top_k": 3, "law_filter": "형법"})
print(f"\n📡 search_law('재산 범죄', law_filter='형법'):\n{result[:300]}")

---
## 문제 4: @tool `interpret_law` — 조문 해석 도구

법률 조문을 쉬운 말로 설명해주는 도구를 구현하세요.

**요구사항:**
- 함수명: `interpret_law`
- 파라미터: `article_id: str` (예: `"민법 제750조"`)
- `SAMPLE_LAW_ARTICLES`에서 해당 조문을 찾음
- `llm.invoke()`로 해석 요청: "일반인이 이해할 수 있는 3-4문장으로 설명"
- 찾지 못하면 `"'{article_id}'를 찾을 수 없습니다."` 반환

**평가기준:**
- 존재하는 조문: LLM 해석 결과 반환
- 존재하지 않는 조문: 에러 메시지 반환
- 결과에 원문 일부가 아닌 '해석'이 포함

In [ ]:
# 문제 4: @tool interpret_law
@tool
def interpret_law(article_id: str) -> str:
    """법률 조문을 일반인이 이해할 수 있게 쉬운 말로 해석합니다.

    Args:
        article_id: 조문 식별자 (예: "민법 제750조", "형법 제329조")
    """
    # ---- 여기에 코드 작성 ----
    # 1) article_id를 "민법 제750조" -> law="민법", art="제750조"로 파싱
    # 2) SAMPLE_LAW_ARTICLES에서 매칭되는 조문 찾기
    # 3) 없으면 에러 메시지 반환
    # 4) llm.invoke()로 해석 요청 (HumanMessage 사용)
    return f"'{article_id}'를 찾을 수 없습니다."

# 테스트
print(interpret_law.invoke({"article_id": "민법 제750조"})[:300])
print()
print(interpret_law.invoke({"article_id": "없는법 제1조"}))

---
## 문제 5: @tool `compare_articles` — 조문 비교 도구

두 법률 조문의 공통점과 차이점을 분석하는 도구를 구현하세요.

**요구사항:**
- 함수명: `compare_articles`
- 파라미터: `article_id1: str, article_id2: str`
- 두 조문을 `SAMPLE_LAW_ARTICLES`에서 찾아 LLM에게 비교 요청
- 결과 형식: "공통점:", "차이점:", "주의사항:" 섹션 포함

**평가기준:**
- 두 조문을 모두 찾으면 비교 결과 반환
- 하나라도 못 찾으면 어떤 것이 없는지 메시지 반환

In [ ]:
# 문제 5: @tool compare_articles
@tool
def compare_articles(article_id1: str, article_id2: str) -> str:
    """두 법률 조문의 공통점과 차이점을 분석합니다.

    Args:
        article_id1: 첫 번째 조문 식별자 (예: "형법 제347조")
        article_id2: 두 번째 조문 식별자 (예: "형법 제329조")
    """
    # ---- 여기에 코드 작성 ----
    # 1) 두 조문 모두 찾기
    # 2) 없는 조문 있으면 에러 반환
    # 3) LLM에게 "공통점 / 차이점 / 주의사항" 형식으로 비교 요청
    return "미구현"

# 테스트
result = compare_articles.invoke({
    "article_id1": "형법 제329조",
    "article_id2": "형법 제347조"
})
print(result[:400])

---
## 문제 6: 에이전트 루프 구현

`bind_tools()`와 `ToolMessage`로 LLM이 도구를 자동 호출하는 에이전트 루프를 구현하세요.

**요구사항:**
- 함수명: `run_legal_agent(query: str, tools: list, max_turns: int = 3) -> str`
- 각 턴에서:
  1. LLM 호출 → 응답에 tool_calls 있는지 확인
  2. tool_calls 있으면 각 도구 실행 → `ToolMessage`로 결과 추가
  3. tool_calls 없으면 최종 답변 반환
- `max_turns` 초과 시 마지막 응답의 content 반환

**평가기준:**
- 질문이 "이혼 사유는?" → `search_law` 자동 호출
- 질문이 "민법 제750조를 쉽게 설명해줘" → `interpret_law` 자동 호출

In [ ]:
# 문제 6: 에이전트 루프
def run_legal_agent(query, tools, max_turns=3):
    """LLM이 도구를 자동 선택하며 실행하는 에이전트 루프."""
    tool_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = [
        SystemMessage(content="당신은 법률 도우미입니다. 필요한 경우 도구를 사용하여 정확한 정보를 제공하세요."),
        HumanMessage(content=query)
    ]

    for turn in range(max_turns):
        # ---- 여기에 코드 작성 ----
        # 1) llm_with_tools.invoke(messages) 호출
        # 2) response.tool_calls 확인
        # 3) tool_calls 없으면 response.content 반환
        # 4) 있으면 각 tool_call 실행 -> ToolMessage(content=..., tool_call_id=...) 추가
        pass

    return "최대 턴 초과"

# 테스트
tools_list = [search_law, interpret_law, compare_articles]
answer = run_legal_agent("이혼 사유에는 어떤 것들이 있어?", tools_list)
print("🤖 답변:", answer[:300])

---
## 문제 7: LegalChatbot 클래스 — 대화 이력 관리

여러 턴 대화가 가능한 `LegalChatbot` 클래스를 구현하세요.

**요구사항:**
- 클래스 메서드: `__init__(tools)`, `chat(user_message: str) -> str`, `reset()`
- 대화 이력을 `self.messages`에 누적 (SystemMessage 포함)
- 각 `chat()` 호출 시 에이전트 루프 실행
- `reset()`은 SystemMessage만 남기고 초기화

**평가기준:**
- 2번째 질문에서 첫 번째 답변을 참고할 수 있는지 (대화 이력 유지)
- `reset()` 호출 시 `len(self.messages) == 1` (SystemMessage만)

In [ ]:
# 문제 7: LegalChatbot 클래스
class LegalChatbot:
    """법률 도메인 대화형 챗봇"""

    def __init__(self, tools):
        self.tools = tools
        self.tool_map = {t.name: t for t in tools}
        self.llm_with_tools = llm.bind_tools(tools)
        self.system_msg = SystemMessage(content=(
            "당신은 법률 도우미입니다. 법률 조문을 검색하고 해석하는 도구를 활용하여 "
            "정확하고 이해하기 쉬운 답변을 제공하세요."
        ))
        self.reset()

    def reset(self):
        """대화 이력 초기화 (SystemMessage만 유지)."""
        # ---- 여기에 코드 작성 ----
        pass

    def chat(self, user_message, max_turns=3):
        """사용자 메시지에 답하며 대화 이력을 유지."""
        # ---- 여기에 코드 작성 ----
        # 1) self.messages에 HumanMessage 추가
        # 2) 에이전트 루프 (문제 6과 유사)
        # 3) 최종 AIMessage를 self.messages에 남기고 content 반환
        return "미구현"

# 테스트
bot = LegalChatbot([search_law, interpret_law, compare_articles])
print("👤 Q1:", "명예훼손 처벌이 어떻게 돼?")
print("🤖:", bot.chat("명예훼손 처벌이 어떻게 돼?")[:200])
print()
print("👤 Q2:", "그럼 허위 사실이면 더 무거워?")
print("🤖:", bot.chat("그럼 허위 사실이면 더 무거워?")[:200])
print(f"\n📝 대화 이력 길이: {len(bot.messages)}")

---
## 문제 8: 추가 도구 — `search_cases` + `explain_term`

판례 검색 도구와 법률 용어 사전 도구를 추가하세요.

**요구사항:**
- `@tool search_cases(keyword: str) -> str` — 샘플 판례 데이터(`SAMPLE_CASES`) 제공됨, 키워드 검색
- `@tool explain_term(term: str) -> str` — 법률 용어 사전(`LEGAL_TERMS`) 제공됨, 용어 설명 반환
- 두 도구 모두 못 찾으면 "관련 정보 없음" 반환

**평가기준:**
- 두 도구 모두 `@tool`로 등록되어 `.name`, `.description` 접근 가능
- `search_cases("이혼")` 시 이혼 관련 판례 반환
- `explain_term("선의의 제3자")` 시 정의 반환

In [ ]:
# 문제 8: 추가 도구 (제공 데이터)
SAMPLE_CASES = [
    {"case_no": "2018다301350", "court": "대법원", "date": "2020-03-26",
     "keyword": ["이혼", "위자료"],
     "summary": "유책배우자의 이혼청구는 원칙적으로 허용되지 않으나 예외 사유에 해당하는 경우 허용될 수 있다는 판례."},
    {"case_no": "2019도14526", "court": "대법원", "date": "2020-07-16",
     "keyword": ["명예훼손", "온라인"],
     "summary": "SNS 게시글이 특정인을 지칭하지 않아도 맥락상 식별 가능하면 명예훼손이 성립한다는 판례."},
    {"case_no": "2017다208748", "court": "대법원", "date": "2019-11-14",
     "keyword": ["사기", "기망"],
     "summary": "부동산 매매에서 중요 사실을 고지하지 않은 경우 부작위에 의한 기망으로 사기죄 성립 가능."},
]

LEGAL_TERMS = {
    "선의의 제3자": "법률관계에서 어떤 사실을 알지 못한 상태에서 새로운 법률관계에 들어온 제3자. 특정 상황에서 보호받는다.",
    "채무불이행": "채무자가 자신의 의무를 이행하지 않거나 불완전하게 이행하는 것.",
    "불법행위": "고의 또는 과실로 타인에게 위법하게 손해를 가하는 행위.",
    "미필적 고의": "결과 발생을 확실히 의도한 것은 아니지만, 발생해도 무방하다고 용인하는 심리 상태.",
    "공소시효": "범죄 후 일정 기간이 지나면 공소를 제기할 수 없게 되는 제도.",
}

# 구현
@tool
def search_cases(keyword: str) -> str:
    """판례를 키워드로 검색합니다. 예: "이혼", "명예훼손", "사기""""
    # ---- 여기에 코드 작성 ----
    return "미구현"

@tool
def explain_term(term: str) -> str:
    """법률 용어의 정의를 설명합니다. 예: "선의의 제3자", "채무불이행""""
    # ---- 여기에 코드 작성 ----
    return "미구현"

# 테스트
print("판례 검색:", search_cases.invoke({"keyword": "이혼"})[:200])
print()
print("용어 설명:", explain_term.invoke({"term": "미필적 고의"}))

---
## 문제 9: 에러 처리 + 면책 조항

도구 실행 실패에 대비한 안전 래퍼와 면책 조항을 구현하세요.

**요구사항:**
- `safe_invoke(tool, args)` 함수: 도구 실행을 try/except로 감싸고 에러 시 `"⚠️ 도구 오류: {에러}"` 반환
- `LEGAL_DISCLAIMER` 상수: "본 답변은 일반적인 정보 제공 목적이며, 구체적인 법률 자문이 아닙니다. 정확한 판단은 전문 변호사와 상담하세요." 같은 문구
- `add_disclaimer(answer: str) -> str`: 답변 끝에 면책 조항을 덧붙임

**평가기준:**
- `safe_invoke`는 어떤 도구 에러도 예외 발생 없이 문자열 반환
- `add_disclaimer`는 입력 답변 + 구분선 + 면책 조항을 포함

In [ ]:
# 문제 9: 에러 처리 + 면책 조항
LEGAL_DISCLAIMER = (
    "⚠️ 본 답변은 일반적인 정보 제공 목적이며, 구체적인 법률 자문이 아닙니다. "
    "정확한 판단은 전문 변호사와 상담하세요."
)

def safe_invoke(tool_fn, args):
    """도구 실행을 안전하게 감쌉니다. 에러 시 메시지 반환."""
    # ---- 여기에 코드 작성 ----
    pass

def add_disclaimer(answer):
    """답변에 면책 조항을 추가합니다."""
    # ---- 여기에 코드 작성 ----
    pass

# 테스트
print("정상 호출:", safe_invoke(search_law, {"query": "살인", "top_k": 1})[:100])
print()
# 일부러 잘못된 인자로 에러 유발
print("에러 호출:", safe_invoke(search_law, {"wrong_key": "test"})[:100])
print()
print(add_disclaimer("이혼은 민법 제840조의 사유가 있을 때 재판상 청구할 수 있습니다."))

---
## 문제 10: 미니 프로젝트 — LegalSearchAgent 통합

Weekend 1의 모든 요소를 통합한 `LegalSearchAgent` 클래스를 완성하세요.

**요구사항:**
- 내부에 모든 도구(`search_law`, `interpret_law`, `compare_articles`, `search_cases`, `explain_term`) 통합
- 메서드: `ask(query: str) -> dict` — 답변 + 사용한 도구 + 소요 시간 반환
- 면책 조항 자동 추가
- 세션 로그 관리: `self.session_log`에 (query, tools_used, time, answer) 기록
- 메서드: `export_session() -> dict` — 전체 세션 요약

**평가기준:**
- `ask()` 반환값에 `answer`, `tools_used`, `elapsed`, `turns` 키 존재
- 답변에 면책 조항 포함
- `export_session()` 결과에 전체 질문 수, 사용 도구 빈도 포함

**→ 이 결과물을 이용해 Weekend 2에서 진행합니다!**

In [ ]:
# 문제 10: LegalSearchAgent 통합
class LegalSearchAgent:
    """모든 법률 도구를 통합한 에이전트 (Weekend 1 최종 결과물)."""

    def __init__(self):
        self.tools = [search_law, interpret_law, compare_articles, search_cases, explain_term]
        self.tool_map = {t.name: t for t in self.tools}
        self.llm_with_tools = llm.bind_tools(self.tools)
        self.session_log = []
        self.system_msg = SystemMessage(content=(
            "당신은 법률 도우미입니다. 주어진 도구를 적극 활용하여 정확한 답변을 제공하세요."
        ))

    def ask(self, query, max_turns=3):
        """질문에 답하며 세션 로그를 기록합니다."""
        start = time.time()
        # ---- 여기에 코드 작성 ----
        # 1) 에이전트 루프 실행 (문제 6~7 참고)
        # 2) tools_used 리스트에 호출한 도구 이름 누적
        # 3) 답변에 면책 조항 추가
        # 4) session_log에 기록
        # 5) {"answer": ..., "tools_used": [...], "elapsed": ..., "turns": ...} 반환
        return {"answer": "미구현", "tools_used": [], "elapsed": 0, "turns": 0}

    def export_session(self):
        """세션 전체 요약을 반환합니다."""
        # ---- 여기에 코드 작성 ----
        return {
            "total_queries": len(self.session_log),
            "tool_frequency": {},
            "total_time": 0,
        }

# 테스트
agent = LegalSearchAgent()
for q in [
    "이혼 사유에는 어떤 것들이 있어?",
    "명예훼손과 허위사실 명예훼손은 어떻게 달라?",
    "미필적 고의가 뭐야?",
]:
    print(f"👤 {q}")
    result = agent.ask(q)
    print(f"🔧 사용 도구: {result['tools_used']}")
    print(f"⏱️ {result['elapsed']}초")
    print(f"🤖 {result['answer'][:200]}...\n")

print("📊 세션 요약:", agent.export_session())

---
## Weekend 1 학습 요약

### 달성한 것들
1. **법률 데이터셋 구축** — 18개 조문 `Document` 변환 + FAISS 색인
2. **5개 도구 구현** — `search_law`, `interpret_law`, `compare_articles`, `search_cases`, `explain_term`
3. **에이전트 루프** — `bind_tools()` + `ToolMessage`로 LLM 자동 도구 선택
4. **대화형 챗봇** — `LegalChatbot`으로 대화 이력 관리
5. **에러 처리 + 면책 조항** — 실전 시스템 요건 충족
6. **`LegalSearchAgent`** — 통합 에이전트 (Weekend 2의 출발점)

### Weekend 2 예고 (LangGraph 에이전트 시스템)
- `LegalAgentState`로 상태 정의 (TypedDict + Annotated messages)
- `StateGraph`로 노드·엣지 구축 (classify → search → analyze → answer)
- 조건부 라우팅으로 질문 유형별 최적 경로
- `MemorySaver`로 영속적 대화 이력 + Human-in-the-Loop